# DeepLense Test 5 - Lens vs Non-Lens Classification

Binary lens classifier trained on `train_lenses` / `train_nonlenses` and evaluated on the held-out `test_lenses` / `test_nonlenses` split.

Strategy: use a compact CNN with class-weighted binary cross-entropy to handle class imbalance, select the best checkpoint by validation AUROC, and report ROC curves and AUROC as the primary task metric. Average precision and PR curves are kept as supplementary imbalance-aware diagnostics.


In [ ]:
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

plt.style.use('seaborn-v0_8-whitegrid')

# Config
SEED = 42
VAL_SIZE = 0.1
BATCH_SIZE = 256
EPOCHS = 80
LR = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 5
NUM_WORKERS = 4

# Paths
cwd = Path.cwd()
ROOT = cwd if (cwd / 'dataset').exists() else cwd / 'test5'
DATA_ROOT = ROOT / 'dataset'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Non-lens', 'Lens']

# Seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin_memory = device.type == 'cuda'

print(f'Device: {device}')
print(f'Data: {DATA_ROOT}')


In [2]:
# Load data
def load_class(folder_name, label):
    paths = sorted((DATA_ROOT / folder_name).glob('*.npy'))
    if not paths:
        raise FileNotFoundError(f'No .npy files in {DATA_ROOT / folder_name}')
    return [(str(p), label) for p in paths]

train_data = load_class('train_nonlenses', 0) + load_class('train_lenses', 1)
test_data = load_class('test_nonlenses', 0) + load_class('test_lenses', 1)

train_paths, train_labels = zip(*train_data)
test_paths, test_labels = zip(*test_data)

# Split train into train/val
train_paths, val_paths, train_labels, val_labels = train_test_split(
    list(train_paths), list(train_labels), 
    test_size=VAL_SIZE, stratify=train_labels, random_state=SEED
)

# Class distribution
train_pos = sum(train_labels)
val_pos = sum(val_labels)
test_pos = sum(test_labels)

print(f'Train: {len(train_labels)} ({train_pos} lens, {len(train_labels)-train_pos} non-lens)')
print(f'Val: {len(val_labels)} ({val_pos} lens, {len(val_labels)-val_pos} non-lens)')
print(f'Test: {len(test_labels)} ({test_pos} lens, {len(test_labels)-test_pos} non-lens)')
print(f'\\nPositive class fraction: train={train_pos/len(train_labels):.4f}, val={val_pos/len(val_labels):.4f}, test={test_pos/len(test_labels):.4f}')

Train: 27364 (1557 lens, 25807 non-lens)
Val: 3041 (173 lens, 2868 non-lens)
Test: 19650 (195 lens, 19455 non-lens)
\nPositive class fraction: train=0.0569, val=0.0569, test=0.0099


In [3]:
# Compute normalization stats (RGB channels)
channel_sum = np.zeros(3)
channel_sq = np.zeros(3)
total_pixels = 0

for path in train_paths:
    img = np.load(path).astype(np.float32)  # Shape: (3, 64, 64)
    channel_sum += img.reshape(3, -1).sum(axis=1)
    channel_sq += (img.reshape(3, -1) ** 2).sum(axis=1)
    total_pixels += img.shape[1] * img.shape[2]

mean = channel_sum / total_pixels
std = np.sqrt(channel_sq / total_pixels - mean ** 2)

print(f'Mean (RGB): {mean}')
print(f'Std (RGB): {std}')

Mean (RGB): [0.27479378 0.19204788 0.10544324]
Std (RGB): [0.17389129 0.15521556 0.11303987]


In [4]:
# Dataset and DataLoaders
class LensDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, i):
        img = torch.from_numpy(np.load(self.paths[i]).astype(np.float32))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[i], dtype=torch.float32)

# Transforms
class RandomRotate90:
    """Rotate by multiples of 90 degrees without interpolation blur."""

    def __call__(self, img):
        k = torch.randint(0, 4, ()).item()
        return torch.rot90(img, k, dims=(-2, -1))

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    RandomRotate90(),
    transforms.Normalize(mean.tolist(), std.tolist())
])

eval_transform = transforms.Normalize(mean.tolist(), std.tolist())

# DataLoaders
train_loader = DataLoader(
    LensDataset(train_paths, train_labels, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory
)
val_loader = DataLoader(
    LensDataset(val_paths, val_labels, eval_transform),
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=pin_memory
)
test_loader = DataLoader(
    LensDataset(test_paths, test_labels, eval_transform),
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=pin_memory
)

In [5]:
# Model
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))
            if stride != 1 or in_ch != out_ch else nn.Identity()
        )
    
    def forward(self, x):
        out = torch.nn.functional.silu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.nn.functional.silu(out + self.shortcut(x))

class LensBinaryResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU()
        )
        self.layer1 = nn.Sequential(ResidualBlock(32, 32), ResidualBlock(32, 32))
        self.layer2 = nn.Sequential(ResidualBlock(32, 64, 2), ResidualBlock(64, 64))
        self.layer3 = nn.Sequential(ResidualBlock(64, 128, 2), ResidualBlock(128, 128))
        self.layer4 = nn.Sequential(ResidualBlock(128, 256, 2), ResidualBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, 1))
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return self.head(x)

model = LensBinaryResNet().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 2,795,297


In [ ]:
# Training with class imbalance handling and AUROC-based model selection
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=pin_memory)
        logits = model(imgs).squeeze(1)
        probs = torch.sigmoid(logits)
        all_labels.append(labels.cpu())
        all_probs.append(probs.cpu())

    y_true = torch.cat(all_labels).numpy()
    y_prob = torch.cat(all_probs).numpy()

    return {
        'auroc': roc_auc_score(y_true, y_prob),
        'ap': average_precision_score(y_true, y_prob),
        'labels': y_true,
        'probs': y_prob,
    }


# Compute pos_weight for class imbalance
neg_count = len(train_labels) - sum(train_labels)
pos_count = sum(train_labels)
pos_weight = torch.tensor([neg_count / pos_count]).to(device)
print(f'Positive class weight: {pos_weight.item():.2f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_score = -np.inf
best_state = copy.deepcopy(model.state_dict())
best_summary = None
patience = 0
history = []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=pin_memory)
        labels = labels.to(device, non_blocking=pin_memory)

        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(imgs)

    # Validate
    val_results = evaluate(model, val_loader)
    scheduler.step(val_results['auroc'])
    lr = optimizer.param_groups[0]['lr']

    history.append({
        'epoch': epoch,
        'train_loss': train_loss / len(train_loader.dataset),
        'val_auroc': val_results['auroc'],
        'val_ap': val_results['ap'],
        'lr': lr,
    })

    print(
        f"Epoch {epoch:02d} | train_loss={train_loss / len(train_loader.dataset):.4f} | "
        f"val_AUROC={val_results['auroc']:.4f} val_AP={val_results['ap']:.4f} | "
        f"lr={lr:.2e}"
    )

    current_score = float(val_results['auroc'])
    if current_score > best_score:
        best_score = current_score
        best_state = copy.deepcopy(model.state_dict())
        best_summary = {
            'auroc': float(val_results['auroc']),
            'ap': float(val_results['ap']),
        }
        torch.save(
            {'model': best_state, 'history': history, 'best_summary': best_summary},
            ARTIFACTS / 'best_model.pt',
        )
        patience = 0
    else:
        patience += 1
        if patience >= EARLY_STOP_PATIENCE:
            print('Early stopping')
            break

model.load_state_dict(best_state)
print(f"Best val AUROC: {best_summary['auroc']:.4f}")
print(f"Best val AP: {best_summary['ap']:.4f}")


In [ ]:
# Plot training curves
df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df['epoch'], df['train_loss'], label='train loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(df['epoch'], df['val_auroc'], label='val AUROC')
axes[1].plot(df['epoch'], df['val_ap'], label='val AP')
axes[1].set_xlabel('Epoch')
axes[1].set_title('Validation Metrics')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Final evaluation on validation and test splits
val_results = evaluate(model, val_loader)
test_results = evaluate(model, test_loader)


def print_split_report(name, results):
    print(f'\n{name} Results:')
    print(f'  AUROC: {results["auroc"]:.4f}')
    print(f'  AP (supplementary): {results["ap"]:.4f}')


print('Official task metric: ROC curve and AUROC on the held-out test set.')
print_split_report('Validation', val_results)
print_split_report('Test', test_results)


In [ ]:
# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves (official task metric)
val_fpr, val_tpr, _ = roc_curve(val_results['labels'], val_results['probs'])
test_fpr, test_tpr, _ = roc_curve(test_results['labels'], test_results['probs'])
axes[0].plot(val_fpr, val_tpr, label=f'Val (AUC={val_results["auroc"]:.3f})')
axes[0].plot(test_fpr, test_tpr, label=f'Test (AUC={test_results["auroc"]:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

# PR curves (supplementary for imbalance discussion)
val_prec, val_rec, _ = precision_recall_curve(val_results['labels'], val_results['probs'])
test_prec, test_rec, _ = precision_recall_curve(test_results['labels'], test_results['probs'])
axes[1].plot(val_rec, val_prec, label=f'Val (AP={val_results["ap"]:.3f})')
axes[1].plot(test_rec, test_prec, label=f'Test (AP={test_results["ap"]:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()

plt.tight_layout()
plt.show()
